In [0]:
# COMMAND ----------
# DBTITLE 1,Importações e Parâmetros
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, when

spark = SparkSession.builder.getOrCreate()

CATALOGO = "bradesco_prod"
TABELA_BRONZE = f"{CATALOGO}.crm_bronze.opportunity_raw"
TABELA_SILVER = f"{CATALOGO}.crm_silver.opportunity_clean"

# COMMAND ----------
# DBTITLE 2,Leitura da Camada Bronze
df_bronze = spark.table(TABELA_BRONZE)

# COMMAND ----------
# DBTITLE 3,Limpeza, Deduplicação e Tratamento de Nulos
df_silver = (
    df_bronze
    # Remove registros duplicados mantendo a chave única
    .dropDuplicates(["Id"])
    # Padronização de strings
    .withColumn("StageName", upper(trim(col("StageName"))))
    # Tratamento de valores nulos
    .withColumn("Amount", when(col("Amount").isNull(), 0.0).otherwise(col("Amount")))
    # Padronização dos nomes de colunas
    .withColumnRenamed("AccountId", "conta_id")
)

# COMMAND ----------
# DBTITLE 4,Gravação na Camada Silver (Delta Lake)
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER)
)

print(f"Camada Silver atualizada com sucesso na tabela: {TABELA_SILVER}")